# RPOE-X — GRPO Training on Google Colab

**OpenEnv Hackathon Submission** | [HF Space](https://bharavi-rpoe-x.hf.space) | [GitHub](https://huggingface.co/spaces/Bharavi/rpoe_x)

Train a Rotary Parking SRE agent using **GRPO** with HF TRL + Unsloth. The agent learns to route
cars across a 5-zone, 20-wheel rotary parking system in HITEC City, Hyderabad — importing all
training logic directly from the `rpoe_x` package.

| Component | Detail |
|-----------|--------|
| Environment | [HF Space](https://bharavi-rpoe-x.hf.space) (RPOE-X server) |
| Training | This Colab notebook (T4 GPU) |
| Model | `Qwen/Qwen2.5-0.5B-Instruct` + LoRA via Unsloth |
| Framework | HF TRL GRPOTrainer + Unsloth 4-bit |

## 1. Install Dependencies

Install the `rpoe_x` package (environment client + training utils) and TRL with Unsloth backend.

In [ ]:
!pip install -q \
    "trl>=0.15.0" \
    "peft" \
    "transformers" \
    "datasets" \
    "huggingface_hub>=0.20.0" \
    "matplotlib" \
    "numpy"

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Install rpoe_x package (environment client + training utilities from HF Space)
!pip install -q "git+https://huggingface.co/spaces/Bharavi/rpoe_x"

## 2. Configuration

Set the HF Space URL, model, and training hyperparameters. Add `HF_TOKEN` to Colab Secrets
(key icon in the left sidebar).

In [ ]:
import os

# HuggingFace token
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except (ImportError, KeyError, ModuleNotFoundError):
    if "HF_TOKEN" not in os.environ:
        print("WARNING: HF_TOKEN not set. Add it via Colab Secrets (left sidebar -> key icon).")

# Environment server (HF Space)
ENV_URL = "https://bharavi-rpoe-x.hf.space"

# Model
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
HUB_REPO = "Bharavi/rpoe-x-qwen-0.5b-grpo"

# Training hyperparameters
NUM_EPISODES    = 60   # episodes for dataset collection
MAX_STEPS       = 300  # GRPO training steps
NUM_GENERATIONS = 4    # completions generated per prompt
MAX_TURNS       = 50   # env steps per collection episode

print(f"Environment : {ENV_URL}")
print(f"Model       : {MODEL_ID}")
print(f"Episodes    : {NUM_EPISODES}")
print(f"Max steps   : {MAX_STEPS}")

## 3. Smoke Test — Verify Environment Connectivity

Connect to the HF Space, reset the environment, and run one action to confirm round-trip works.

In [ ]:
from rpoe_x import ParkingEnv, ParkingAction

print(f"Connecting to {ENV_URL} ...")

async with ParkingEnv(base_url=ENV_URL) as env:
    reset_result = await env.reset()
    obs = reset_result.observation
    print("Connected!\n")
    print(f"Zone occupancy : {[round(x, 2) for x in obs.zone_occupancy]}")
    print(f"Queue lengths  : {obs.zone_queue_lengths}")
    print(f"Time of day    : {obs.time_of_day:.3f}")

    result = await env.step(ParkingAction(zone_id=2, wheel_id=0))
    print(f"\nSmoke test step (reward={result.reward:.3f})")
    print(f"  Zone occupancy: {[round(x, 2) for x in result.observation.zone_occupancy]}")

print("\nSmoke test passed. Environment is ready for training.")

## 4. Import Training Utilities from Package

All training logic (system prompts, observation formatters, action parser, reward functions,
episode collector, reward plotter) is imported directly from `rpoe_x.training.train` —
no duplication between notebook and package.

In [ ]:
import logging
import random
from datetime import datetime
from pathlib import Path

from datasets import Dataset
from transformers import AutoTokenizer
from unsloth import FastLanguageModel, PatchDPOTrainer
from trl import GRPOConfig, GRPOTrainer

# Import ALL training utilities from the rpoe_x package
from rpoe_x.training.train import (
    ORCH_SYSTEM,
    ZONE_SYSTEM,
    format_orch_obs,
    format_zone_obs,
    parse_action,
    format_reward,
    routing_reward,
    wheel_reward,
    collect_episode,
    plot_rewards,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

print("ORCH_SYSTEM (first 80 chars):")
print(f"  {ORCH_SYSTEM[:80]}...")
print(f"\nImported: format_orch_obs, format_zone_obs, parse_action")
print(f"          format_reward, routing_reward, wheel_reward")
print(f"          collect_episode, plot_rewards")

## 5. GRPO Training Setup

Load the model with Unsloth, collect the training dataset from the environment,
then configure the GRPO trainer.

In [ ]:
import torch

# ── Tokenizer ─────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── Model (Unsloth 4-bit quantisation for T4 memory efficiency) ───────────────
model, _ = FastLanguageModel.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = 512,
    dtype          = None,
    load_in_4bit   = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 16,
    use_gradient_checkpointing = "unsloth",
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Model : {MODEL_ID}")
print(f"Params: {trainable:,} trainable / {total:,} total ({100 * trainable / total:.2f}%)")

# ── Collect training dataset ──────────────────────────────────────────────────
# collect_episode applies tokenizer.apply_chat_template so prompt is a string.
# Variable-length lists (zone_queue_lengths, wheel_occupancy) are JSON-serialised
# to avoid TRL batch-collation errors.
print(f"\nCollecting {NUM_EPISODES} episodes from {ENV_URL} ...")
all_rows = []

async with ParkingEnv(base_url=ENV_URL) as env:
    for ep in range(NUM_EPISODES):
        rows = await collect_episode(env, tokenizer, MAX_TURNS)
        all_rows.extend(rows)
        if (ep + 1) % 10 == 0:
            logger.info(f"Episodes {ep - 8}–{ep + 1} done ({len(all_rows):,} examples so far)")

random.shuffle(all_rows)
dataset = Dataset.from_list(all_rows)

orch_n = sum(1 for r in dataset if r["agent_role"] == "orchestrator")
zone_n = len(dataset) - orch_n
print(f"\nDataset: {len(dataset):,} examples ({orch_n:,} orchestrator, {zone_n:,} zone)")
print(f"Prompt type: {type(dataset[0]['prompt']).__name__} — must be str")

# ── Output directory ──────────────────────────────────────────────────────────
timestamp  = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_dir = f"outputs/rpoe-x-grpo-{timestamp}"
Path(output_dir).mkdir(parents=True, exist_ok=True)
print(f"Output : {output_dir}")

In [ ]:
# ── GRPO Config ───────────────────────────────────────────────────────────────
PatchDPOTrainer()
FastLanguageModel.for_training(model)

grpo_config = GRPOConfig(
    output_dir                  = output_dir,
    learning_rate               = 5e-6,
    lr_scheduler_type           = "cosine",
    warmup_steps                = 20,
    max_steps                   = MAX_STEPS,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    num_generations             = NUM_GENERATIONS,
    max_prompt_length           = 384,
    max_completion_length       = 128,
    temperature                 = 0.9,
    logging_steps               = 10,
    save_strategy               = "no",
    report_to                   = "none",
    fp16                        = True,
    seed                        = 42,
)

# ── Create Trainer (reward functions imported from package) ───────────────────
trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [format_reward, routing_reward, wheel_reward],
    args             = grpo_config,
    train_dataset    = dataset,
)

print("GRPOTrainer initialized")

## 6. Train!

Launch GRPO training. Each step: sample prompt from dataset → generate
`NUM_GENERATIONS` completions → score with reward functions → GRPO gradient update.

In [ ]:
print("Starting GRPO training ...")
print(f"  Model       : {MODEL_ID}")
print(f"  Max steps   : {MAX_STEPS}")
print(f"  Generations : {NUM_GENERATIONS}")
print(f"  Dataset     : {len(dataset):,} examples")
print()

trainer.train()
trainer.save_model(output_dir)

print(f"\nModel saved to {output_dir}")

## 7. Reward Curves

Visualise training progress using `plot_rewards` from the package.

In [ ]:
plot_rewards(trainer, save_path=f"{output_dir}/reward_curve.png")

## 8. Push to Hub (Optional)

Upload the trained LoRA adapter to HuggingFace Hub.

In [ ]:
# Uncomment to push:
# trainer.push_to_hub(repo_id=HUB_REPO)
# print(f"Model pushed to https://huggingface.co/{HUB_REPO}")